In [ ]:
import os
import json
import pandas as pd
import random
import arc
import importlib
importlib.reload(arc)
from arc import ARCSolver_MoE
from task_clusterization.clusterization import save_cluster_model, classify_cluster
from arc import ARCSolver

In [2]:
def load_data(base_dir):
    print("\n▶ Loading training datasets\n")

    filenames = os.listdir(base_dir)
    data_files = [os.path.join(base_dir, p) for p in filenames if ".json" in p]

    dataset = []
    for fn in data_files:
        with open(fn) as fp:
            data = json.load(fp)
        dataset.append(data)

    filenames = [fn.split(".")[0] for fn in filenames]
    arc_data = []
    print(sum([len(data) for data in dataset]))
    for task_idx, data in enumerate(dataset):
        N = len(data)
        for i in range(0, N, 4):
            if i + 3 >= N:
                continue
            train_grids = [grid for grid in data[i:i+3]]
            test_grids = [grid for grid in data[i+3:i+4]]

            arc_data.append({
            'train': train_grids,
            'test': test_grids
        })

    random.seed(42) 
    random.shuffle(arc_data)
    print("\n▶ Successfully loaded training datasets: ", len(arc_data), "\n")
    return arc_data

In [5]:
def split_data(dataset, cluster_model_dir: str, n_clusters: int):
    """
    주어진 dataset을 classify_cluster 결과에 따라 n_clusters개의 그룹으로 분할합니다.

    Args:
        dataset (list of dict): 각 요소가 적어도 'train' 키를 가진 dict로, 분류할 데이터 예시를 포함합니다.
        cluster_model_dir (str): classify_cluster 함수가 사용할 모델 디렉터리 경로입니다.
        n_clusters (int): 클러스터 개수 (0부터 n_clusters-1 까지 레이블이 나타납니다).

    Returns:
        List[List[dict]]: i번째 리스트에는 레이블 i로 분류된 데이터 포인트들을 담고 있습니다.
    """
    # 레이블별로 데이터를 담을 빈 리스트 초기화
    clusters = [[] for _ in range(n_clusters)]

    for idx, item in enumerate(dataset):
        # 데이터 예시는 'train' 키 아래에 있다고 가정
        train_example = item['train']
        label,_ = classify_cluster(cluster_model_dir, train_example)

        # 레이블 유효성 검사
        if not (0 <= label < n_clusters):
            raise ValueError(f"인덱스 {idx}의 예제에 대해 잘못된 레이블이 반환되었습니다: {label}")

        clusters[label].append(item)

    return clusters

In [14]:
data_path = "../dataset"
# aug_data_path = "../dataset_generated"

use_pretrained_model = True

train_dataset = load_data(data_path)
train_dataset = train_dataset[:100] # For testing
DATASET_DIR_LIST = [data_path]
cluster_model_dir = "artifacts/cluster_model"
n_clusters = 6
save_cluster_model(DATASET_DIR_LIST, cluster_model_dir, n_clusters)
train_dataset_splitted = split_data(train_dataset,cluster_model_dir,n_clusters)

Silhouette Score (k=6): 0.578
Cluster 0 (50 tasks): ['../dataset/e3497940', '../dataset/ff28f65a', '../dataset/90c28cc7', '../dataset/23b5c85d', '../dataset/77fdfe62', '../dataset/1c786137', '../dataset/1190e5a7', '../dataset/6430c8c4', '../dataset/9ecd008a', '../dataset/5614dbcf', '../dataset/99b1bc43', '../dataset/2dc579da', '../dataset/d9fac9be', '../dataset/5daaa586', '../dataset/f2829549', '../dataset/7c008303', '../dataset/1b2d62fb', '../dataset/2dee498d', '../dataset/dae9d2b5', '../dataset/dc0a314f', '../dataset/44f52bb0', '../dataset/94f9d214', '../dataset/239be575', '../dataset/d10ecb37', '../dataset/0b148d64', '../dataset/a68b268e', '../dataset/de1cd16c', '../dataset/75b8110e', '../dataset/d4469b4b', '../dataset/47c1f68c', '../dataset/5bd6f4ac', '../dataset/ff805c23', '../dataset/662c240a', '../dataset/4be741c5', '../dataset/88a62173', '../dataset/6773b310', '../dataset/25d8a9c8', '../dataset/0d3d703e', '../dataset/c909285e', '../dataset/995c5fa3', '../dataset/3428a4f5', '../

In [5]:
token = os.environ.get("HF_TOKEN", None)
solver = ARCSolver_MoE(token=token)
solver.train(train_dataset_splitted, pretrained=use_pretrained_model)


>>> Load model...

==((====))==  Unsloth 2025.4.7: Fast Llama patching. Transformers: 4.51.3. vLLM: 0.8.5.
   \\   /|    NVIDIA TITAN Xp. Num GPUs = 1. Max memory: 11.897 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 6.1. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

>>> Successfully loaded model


>>> Shrink Embedding dimensions...


>>> Successfully shrinked embedding dimensions


>>> Expert Fin-tuning of 0/2.


>>> Load a pretrained model...



Unsloth 2025.4.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM

>>> Successfully loaded a pretrained model


>>> Successfully preparing PEFT/LoRA



Prepare training dataset...:   0%|          | 0/32 [00:00<?, ?it/s]


>>> Successfully prepared training dataset

trainable params: 389,507,072 || all params: 3,208,497,152 || trainable%: 12.1399

>>> Starting fine-tuning...



...


>>> Successfully prepared training dataset

trainable params: 389,507,072 || all params: 3,208,497,152 || trainable%: 12.1399

>>> Starting fine-tuning...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 68 | Num Epochs = 2 | Total steps = 16
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 389,507,072/3,000,000,000 (12.98% trained)


Step,Training Loss
10,0.020200


/home/student/.conda/envs/project/lib/python3.11/site-packages/peft/utils/save_and_load.py:250: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(



>>> Training complete. Saving adapter to artifacts/checkpoint-final-test-05-22_MoE/1

>>> Fin-tuning of 1/2 finished and adapter saved.



In [5]:
import importlib
import task_clusterization.clusterization
# importlib.reload(task_clusterization.clusterization)
from task_clusterization.clusterization import save_cluster_model, classify_cluster
# from task_clusterization.task import Task
DATASET_DIR_LIST = ["../dataset_generated"]#,'../dataset_generated'
# DATASET_DIR_LIST = ["../dataset"]#,'../dataset_generated/']
model_dir = "artifacts/cluster_model"

n_clusters = 6
save_cluster_model(DATASET_DIR_LIST, model_dir, n_clusters)

Silhouette Score (k=6): 0.552
Cluster 0 (184 tasks): ['../dataset_generated/self_instruct_code_fewshot_4_gpt-4o-mini_temp0.70_maxtokens2048_briefcommon_description_file_self_instruct_descriptions_fewshot_70_gpt-4o-mini_temp0.70_maxtokens1024_rng5_used_concepts_generated_problems_19', '../dataset_generated/self_instruct_code_fewshot_4_gpt-4o-mini_temp0.70_maxtokens2048_briefcommon_description_file_self_instruct_descriptions_fewshot_70_gpt-4o-mini_temp0.70_maxtokens1024_rng10_used_concepts_generated_problems_35', '../dataset_generated/self_instruct_code_fewshot_4_gpt-4o-mini_temp0.70_maxtokens2048_briefcommon_description_file_self_instruct_descriptions_fewshot_70_gpt-4o-mini_temp0.70_maxtokens1024_rng3_used_concepts_generated_problems_23', '../dataset_generated/self_instruct_code_fewshot_4_gpt-4o-mini_temp0.70_maxtokens2048_briefcommon_description_file_self_instruct_descriptions_fewshot_70_gpt-4o-mini_temp0.70_maxtokens1024_rng1_used_concepts_generated_problems_28', '../dataset_generated

In [8]:
label = classify_cluster(model_dir,train_dataset[8]['train'])
print(label)

[[0.         0.89591267 0.69240214]]
3


/home/student/.conda/envs/project/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
